# Plant Disease Classification
Classifying tomato, potato and pepper leaf diseases from the PlantVillage dataset with a frozen MobileNetV3Large base.

In [ ]:
import os
import random
import shutil
import time
import numpy as np
import tensorflow as tf
from keras import layers, models
from keras.applications import MobileNetV3Large
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import kagglehub

In [ ]:
path = kagglehub.dataset_download("emmarex/plantdisease")
directory = os.path.join(path, "PlantVillage")

## Looking at the data

In [ ]:
# the download has a duplicate PlantVillage folder inside it, so skip that
class_names = sorted([d for d in os.listdir(directory)
                      if os.path.isdir(os.path.join(directory, d)) and d != "PlantVillage"])

counts = {}
for c in class_names:
    files = [f for f in os.listdir(os.path.join(directory, c)) if f.lower().endswith((".jpg", ".jpeg"))]
    counts[c] = len(files)
    print(f"{c}: {len(files)}")
print("Total:", sum(counts.values()))

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(list(counts.keys()), list(counts.values()))
plt.title("Images per class (full dataset)")
plt.xlabel("Number of images")
plt.tight_layout()
plt.show()

## Subsample and split
Training on all 20,000 images is too slow on my laptop's CPU, so I take up to 200 images per class. 15% of each class goes into a test folder that the model never sees during training, and the rest is split 82/18 into train and validation (roughly 70/15/15 overall).

In [ ]:
PER_CLASS = 200
TEST_FRAC = 0.15
random.seed(42)

if os.path.exists("data"):
    shutil.rmtree("data")

for c in class_names:
    files = sorted([f for f in os.listdir(os.path.join(directory, c)) if f.lower().endswith((".jpg", ".jpeg"))])
    random.shuffle(files)
    files = files[:PER_CLASS]
    n_test = int(len(files) * TEST_FRAC)
    for i, f in enumerate(files):
        split = "test" if i < n_test else "train_val"
        os.makedirs(os.path.join("data", split, c), exist_ok=True)
        shutil.copy(os.path.join(directory, c, f), os.path.join("data", split, c, f))

for split in ["train_val", "test"]:
    total = sum(len(os.listdir(os.path.join("data", split, c))) for c in class_names)
    print(split, total)

In [ ]:
BATCH = 16
SIZE = (224, 224)

def data_loading(subset):
    dataset = image_dataset_from_directory(
        "data/train_val",
        labels = "inferred",
        label_mode = "categorical",
        validation_split = 0.18,
        subset = subset,
        shuffle = True,
        seed = 123,
        batch_size = BATCH,
        image_size = SIZE,
    )
    return dataset

train_ds = data_loading("training")
val_ds = data_loading("validation")

# test set is not shuffled so predictions line up with the labels
test_ds = image_dataset_from_directory(
    "data/test",
    labels = "inferred",
    label_mode = "categorical",
    shuffle = False,
    batch_size = BATCH,
    image_size = SIZE,
)
class_names = train_ds.class_names
num_classes = len(class_names)

In [ ]:
# one example of each class
rep_images = {c: None for c in class_names}
for img, label in train_ds.unbatch():
    c = class_names[np.argmax(label.numpy())]
    if rep_images[c] is None:
        rep_images[c] = img.numpy().astype("uint8")
    if all(v is not None for v in rep_images.values()):
        break

plt.figure(figsize=(15, 9))
for i, c in enumerate(class_names):
    plt.subplot(3, 5, i + 1)
    plt.imshow(rep_images[c])
    plt.title(c.replace("_", " "), fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()